# CFP_Chronos_Forecasts

Generates 1-step-ahead VaR forecasts using Amazon Chronos-v1 Small (46M) and Mini (20M) foundation models. Computes 1000 ancestral samples per day, extracts VaR quantiles.

**Paper:** Pele, Lessmann, Hardle (2026)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

DATA_DIR = Path('../cfp_ijf_data')
RET_DIR = DATA_DIR / 'returns'

ASSETS = [
    'SP500','STOXX','GDAXI','CACT','FTSE100','ICLN','NIKKEI','HSI','BOVESPA','NIFTY','ASX200',
    'CBU0','TLT','IBGL','DJCI','GOLD','WTI','NATGAS','BTC','ETH','EURUSD','GBPUSD','USDJPY','AUDUSD'
]
ALPHAS = [0.01, 0.025, 0.05, 0.10]
CONTEXT = 512
N_SAMPLES = 1000

def load_returns(asset):
    df = pd.read_csv(RET_DIR / f'{asset}.csv', parse_dates=['date']).set_index('date').sort_index()
    df = df[df['log_return'].abs() <= 0.50]
    return df['log_return']

print(f'Assets: {len(ASSETS)}, Context: {CONTEXT}, Samples: {N_SAMPLES}')

MODELS = [('chronos_small', 'amazon/chronos-t5-small', 'Chronos-Small'), ('chronos_mini', 'amazon/chronos-t5-mini', 'Chronos-Mini')]

In [ ]:
import torch
from chronos import ChronosPipeline

for model_slug, model_id, label in MODELS:
    print(f'\n=== {label} ({model_id}) ===')
    pipeline = ChronosPipeline.from_pretrained(model_id, dtype=torch.float32, device_map='cpu')
    out_dir = DATA_DIR / model_slug
    out_dir.mkdir(exist_ok=True)

    for asset in tqdm(ASSETS, desc=label):
        ret = load_returns(asset)
        n = len(ret)
        vals = ret.values
        dates = ret.index
        records = []
        for t in range(CONTEXT, n):
            context = torch.tensor(vals[t-CONTEXT:t], dtype=torch.float32).unsqueeze(0)
            samples = pipeline.predict(context, prediction_length=1, num_samples=N_SAMPLES)
            s = samples[0, :, 0].numpy()
            row = {'date': dates[t]}
            for alpha in ALPHAS:
                row[f'VaR_{alpha}'] = np.percentile(s, alpha * 100)
            row['mean'] = s.mean()
            row['std'] = s.std()
            records.append(row)
        df_out = pd.DataFrame(records).set_index('date')
        df_out.to_parquet(out_dir / f'{asset}.parquet')
    print(f'  {label}: {len(ASSETS)} assets saved')

In [ ]:
# Verify outputs
from pathlib import Path
for model_slug, _, label in MODELS:
    out_dir = DATA_DIR / model_slug
    files = list(out_dir.glob('*.parquet'))
    print(f'{label}: {len(files)} parquet files')